# MPC Helpers Quick Tutorial

This notebook is the short beginner tutorial for the public MPC helper functions in `EOptInterface.jl`.

The goal is not to teach all of MPC theory.
The goal is to help a new user answer three practical questions:
- Which helper should I call first?
- What data shape does each helper expect?
- How do the helpers connect during a normal MPC workflow?

A good mental model is:
1. build JuMP state trajectories;
2. copy the current plant state into those trajectories;
3. warm-start the control variables;
4. log plant data, predictions, and objective terms;
5. use diagnostics only if something looks wrong.


In [ ]:
using Pkg

repo_root = isdir(joinpath(pwd(), "src")) ? pwd() : normpath(joinpath(pwd(), ".."))
Pkg.activate(repo_root)

using EOptInterface
using JuMP
using ModelingToolkit
using ModelingToolkit: t_nounits as t, D_nounits as D


## 1. Start From A Tiny Toy System

We start with a very small MTK model on purpose.
That keeps the helper functions easy to see.

The core idea is simple:
- the MTK model defines the symbolic state names;
- the ODE integrator stores the numeric state values;
- the helper functions translate between those two views.

Once that idea is clear here, the same pattern carries over to larger WRRF models.


In [ ]:
@parameters u = 0.0 d = 1.0
ModelingToolkit.@variables x(t) = 0.0 y(t) = 0.0
@named toy = ODESystem([
    D(x) ~ -x + y + u,
    D(y) ~ -0.5 * y + d,
], t, [x, y], [u, d])

state_vars = collect(unknowns(toy))
x_key = state_vars[1]
y_key = state_vars[2]
index_map = make_state_index_map(state_vars)

struct FakeIntegrator
    t::Float64
    u::Vector{Float64}
    ps::Dict{Any, Float64}
end

integ = FakeIntegrator(3.5, [0.4, 1.2], Dict(toy.u => 0.6, toy.d => 1.0))

(
    state_vars = state_vars,
    current_by_sys = current_state_map(integ, toy),
    current_by_map = current_state_map(integ, index_map),
)


## 2. Build State Trajectories And Synchronize Initial Conditions

This is the first low-level MPC step.
Before an optimizer can predict the future, it needs one JuMP trajectory for each plant state.
It also needs the first point of each trajectory to match the current plant condition.

So this section does three things:
1. build one JuMP trajectory for each state;
2. update the first point from the latest plant value;
3. optionally use that same value as a simple warm start across the horizon.

This is the bridge between the plant model and the optimizer model.


In [ ]:
model = Model()
x_vars, c_ic = build_state_trajs_from_vars!(model, toy, state_vars, 5; lb = -2.0, ub = 3.0, rhs0 = 0.0)

state_values = Dict(x_key => 0.4, y_key => 1.2)
sync_state_trajectories!(c_ic, x_vars, state_values; set_start = true, fill_horizon = true)
sync_report = check_ic_sync(state_values, c_ic)

(
    trajectory_keys = collect(keys(x_vars)),
    first_rhs_x = JuMP.normalized_rhs(c_ic[x_key]),
    horizon_start_x = [JuMP.start_value(v) for v in x_vars[x_key]],
    sync_report = sync_report,
)


## 2b. Read MPC Names The Way Humans Do

A common beginner problem is not the math.
It is finding the right `VariableRef` later when you want to inspect a variable or add a constraint.

These helpers separate two different ideas:
- a human-readable display name;
- a solver-safe JuMP base name.

Use them to answer questions like:
- "What should this MTK state be called in logs?"
- "How do I find the matching JuMP trajectory again later?"


In [ ]:
canonical_x = canonical_system_unknown(toy, toy.x)
canonical_u = canonical_system_parameter(toy, toy.u)
canonical_d = canonical_system_parameter(toy, toy.d)

(
    canonical_x = canonical_x,
    display_x = display_mpc_name(canonical_x),
    state_base_x = state_trajectory_base_name(toy, canonical_x),
    control_base_u = resolve_mpc_base_name(canonical_u; suffix = "u"),
    stage_base_d = resolve_mpc_base_name(canonical_d; suffix = "stage"),
)


## 3. Warm-Start The Control Sequence And Bound The First Move

These helpers update the control side of the MPC problem.

In a real receding-horizon loop, the next solve is usually similar to the last one.
So we do two useful things:
- reuse the previous control sequence as a warm start;
- limit how far the first new move can jump from the currently applied control.

Those two steps often make the online solve more robust and easier to interpret.


In [ ]:
control_model = Model()
@variable(control_model, utraj[1:5] >= 0.0)

warm_start_with_constant!(utraj, 0.8)
constant_start = [JuMP.start_value(ui) for ui in utraj]

shift_warm_start!(utraj, [0.8, 0.7, 0.6, 0.5, 0.4])
shifted_start = [JuMP.start_value(ui) for ui in utraj]

first_move_bounds = set_first_move_bounds!(utraj, 0.65; lower = 0.0, upper = 2.0, delta_max = 0.15)

(
    constant_start = constant_start,
    shifted_start = shifted_start,
    first_move_bounds = first_move_bounds,
)


## 4. Keep A Standard Closed-Loop Log

For small MPC studies, keep one standard log from the start.
Good logging saves a lot of debugging time later.

The basic pattern is:
1. build the log with `make_mpc_log`;
2. save the initial point with `seed_mpc_log!`;
3. save each new plant state with `log_mpc_state!`;
4. save each new prediction with `record_mpc_prediction!`;
5. save each objective summary with `record_mpc_metrics!`.

This creates one consistent record of what the plant did and what the optimizer predicted.


In [ ]:
logctx = make_mpc_log(state_vars; control_keys = [:u], predicted_keys = [x_key], metric_keys = [:objective, :move_u])

seed_mpc_log!(logctx, state_values, 0.0; control_values = Dict(:u => 0.65))

integ1 = FakeIntegrator(1.0, [0.55, 1.10], Dict(toy.u => 0.70, toy.d => 1.0))
log_mpc_state!(logctx, integ1; control_values = Dict(:u => 0.70))
record_mpc_prediction!(logctx, 1.0, Dict(x_key => [0.55, 0.68, 0.79]))
record_mpc_metrics!(logctx, Dict(:objective => 1.8, :move_u => 0.05))

integ2 = FakeIntegrator(2.0, [0.69, 1.00], Dict(toy.u => 0.52, toy.d => 1.0))
log_mpc_state!(logctx, integ2; control_values = Dict(:u => 0.52))
record_mpc_prediction!(logctx, 2.0, Dict(x_key => [0.69, 0.77, 0.83]))
record_mpc_metrics!(logctx, Dict(:objective => 1.1, :move_u => 0.02))

prediction_error_report = compute_step_prediction_errors(logctx, x_key)

(
    logged_times = logctx.ts,
    state_history = logctx.Xhist,
    control_history = logctx.Controlhist,
    metric_history = logctx.Metrichist,
    prediction_error_report = prediction_error_report,
)


## 4b. Optional: Query Exact Trajectories And Print Live MPC Status

This section is the bridge from helper functions to a real MPC script.

Here the main questions are:
- How do I ask for the exact state or control trajectory I want?
- How do I print one short status line while the closed-loop run is happening?

The lookup helpers answer the first question.
`print_tracking_status(...)` answers the second.
Together they make the online MPC loop much easier to inspect.


In [ ]:
ipopt_available = try
    @eval using Ipopt
    true
catch err
    @warn "Skipping the optional tracking-MPC status demo because Ipopt is not available in the active environment." exception = (err, catch_backtrace())
    false
end

if ipopt_available
    status_model = Model(Ipopt.Optimizer)
    set_silent(status_model)

    status_ctrl = build_tracking_mpc(
        status_model,
        toy;
        control_specs = [
            MPCControlSpec(sym = toy.u, lower = 0.0, upper = 2.0, delta_max = 0.25, move_weight = 0.1),
        ],
        output_specs = [
            MPCOutputSpec(sym = toy.x, setpoint = 1.0, track_weight = 1.0, lower_soft = 0.0, upper_soft = 2.0, slack_weight = 10.0),
        ],
        stage_param_defaults = Dict(toy.d => fill(1.0, 4)),
        config = TrackingMPCConfig(PH = 3, CH = 2, dt = 1.0, integrator = "IE", system_kind = :ode),
    )

    x_traj = state_traj(status_ctrl, "x")
    u_traj = control_traj(status_ctrl, toy.u)
    d_traj = stage_param_traj(status_ctrl, "d_stage")

    @constraint(status_model, x_traj[2] <= 2.5)

    status_result = solve_tracking_mpc!(
        status_ctrl,
        Dict(toy.x => 0.4, toy.y => 1.2),
        Dict(toy.u => 0.6);
        show_status = true,
        status_time = integ.t,
        status_output_syms = [toy.x],
        status_control_syms = [toy.u],
    )

    (
        x_var_name = JuMP.name(x_traj[1]),
        u_var_name = JuMP.name(u_traj[1]),
        d_var_name = JuMP.name(d_traj[1]),
        display_x = display_mpc_name(canonical_system_unknown(toy, toy.x)),
        first_control_move = status_result.controls[toy.u][1],
        first_prediction = status_result.predictions[toy.x][1],
        metrics = status_result.metrics,
    )
else
    "Ipopt not available in the active environment; skip this optional section."
end


## 5. Use Diagnostics Only When Something Looks Wrong

Most new users do not need these helpers on day one.
They become useful when the model does not solve, constraints conflict, or variable naming becomes confusing.

So think of this section as a repair kit.
Do not start here.
Come back here when a normal workflow stops behaving as expected.


In [ ]:
summary_model = Model()
@variable(summary_model, q >= 0.0)
@constraint(summary_model, q == 2.0)

(
    accepted_statuses = default_mpc_accepted_statuses(),
    status_ok = is_accepted_mpc_status(JuMP.MOI.LOCALLY_SOLVED),
    objective_before_solve = objective_value_or_nan(summary_model),
    solve_summary = summarize_mpc_solve(summary_model),
    conflicts = find_conflicts(summary_model),
    ic_uniqueness = summarize_ic_uniqueness(summary_model),
)


## 6. Optional: Path Helpers For Nested WRRF Models

These helpers matter more in large flowsheets than in tiny toy models.

They help when state names carry subsystem paths such as:
- `reactor1₊S_O(t)`;
- `clarifier₊outlet_stream₊S_N2O(t)`.

The package still needs readable names and reliable lookups in those larger models.
These helpers handle that translation.


In [ ]:
struct FakeNode
    name::Symbol
    children::Dict{Symbol, Any}
end

Base.getproperty(node::FakeNode, sym::Symbol) =
    sym === :name || sym === :children ? getfield(node, sym) : node.children[sym]

outlet = FakeNode(Symbol("clarifier₊outlet_stream"), Dict(:S_N2O => :clf_n2o))
reactor1 = FakeNode(:reactor1, Dict(:S_O => :r1_so, :S_NH => :r1_snh))
clarifier = FakeNode(:clarifier, Dict(:outlet_stream => outlet))
influent = FakeNode(:Influent, Dict(:COD => :influent_cod))
fake_sys = FakeNode(:sys, Dict(:reactor1 => reactor1, :clarifier => clarifier, :Influent => influent))

path_vars = [
    "reactor1₊S_O(t)",
    "clarifier₊outlet_stream₊S_N2O(t)",
    "reactor1₊S_NH(t)",
    "Influent₊COD(t)",
]

(
    grouped = group_state_symbols(path_vars),
    pretty_subsystems = pretty_subsystems(fake_sys, collect_subsystems(fake_sys, path_vars)),
)


## Recommended Next Files

After this notebook, use the examples in this order:
- open `examples/tracking_mpc_demo.jl` for the smallest complete tracking MPC path;
- open `examples/dmc_registration_demo.jl` for the light DMC path;
- open `examples/dae_registration_demo.jl` for the low-level DAE registration path;
- open `examples/ndmc_conductivity_mpc_demo.jl` for a larger closed-loop case study.

That order moves from the smallest public example to the most complete one.
